# channel_lmm_anatomical.ipynb\n\n**Purpose:** Channel-level LMM to identify which specific fNIRS channels\nshow a significant group × session interaction in HbO beta across training.\n\n**Inputs:**\n- subjstats_completo.xlsx — fNIRS GLM betas from MATLAB SubjStats\n\n**Outputs:**\n- channel_lmm_hbo.xlsx — 15 channels with coef, p, q-FDR; 6 significant (q<0.05)\n- Forest plot with anatomical labels (PNG)\n\n**Model:** eta ~ group * sessao_num + (1|subject) [ML] applied to each of 15 HbO channels\n- Multiple comparisons correction: Benjamini-Hochberg FDR (q < 0.05)\n- Anatomical labelling via 10-10 system correspondence\n\n**Library:** statsmodels 0.14.6 (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# Channel-Level LMM with Anatomical Labels
**Project SESI | Input: subjstats_all.xlsx**

- One LMM per channel: `beta ~ group * sessao_num + (1|subject)`
- Results labeled with anatomical region (10-10 system)
- Switch chromophore in Block 1 — everything propagates automatically

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration
**Change `CHROMOPHORE` to switch between `'StO2'`, `'hbo'`, `'hbr'`, `'hbt'`**

In [ ]:
# ── CHANGE THIS LINE TO SWITCH CHROMOPHORE ────────────────────
CHROMOPHORE = 'hbo'   # options: 'StO2' | 'hbo' | 'hbr' | 'hbt'
# ─────────────────────────────────────────────────────────────

# Update this path to match your local data directory
PATH_IN        = r'../data/subjstats_individuais.xlsx'
# Update this path to match your local data directory
BASE_OUT       = r'../results/channel_lmm'
PATH_OUT_EXCEL = f'{BASE_OUT}_{CHROMOPHORE}.xlsx'
PATH_OUT_FIG   = f'{BASE_OUT}_forest_{CHROMOPHORE}.png'
PATH_OUT_TRAJ  = f'{BASE_OUT}_trajectory_{CHROMOPHORE}.png'

# ── Anatomical mapping: S-D → (10-10 name, anatomical region, ROI) ───
CHANNEL_MAP = {
    # FRONTAL
    'S1-D1': ('F3-F5',    'L Middle Frontal Gyrus (60%)',    'FRONTAL'),
    'S1-D2': ('F3-F1',    'L Superior Frontal Gyrus (31%)',  'FRONTAL'),
    'S2-D1': ('AF7-F5',   'L IFG p.Triangularis (54%)',      'FRONTAL'),
    'S2-D3': ('AF3-Fp1',  'L Superior Frontal Gyrus (22%)',   'FRONTAL'),
    'S3-D1': ('AF3-F5',   'L Middle Frontal Gyrus (66%)',     'FRONTAL'),
    'S3-D3': ('AF3-Fp1',  'L Superior Frontal Gyrus (56%)',  'FRONTAL'),
    'S4-D1': ('AF3-F1',   'L Superior Frontal Gyrus (55%)',  'FRONTAL'),
    # TEMPORAL
    'S5-D4': ('P7-TP7',   'L Middle Temporal Gyrus (41%)',   'TEMPORAL'),
    'S5-D5': ('P7-P9',    'L Inferior Temporal Gyrus (29%)', 'TEMPORAL'),
    'S5-D6': ('P7-P5',    'L Middle Temporal Gyrus (47%)',   'TEMPORAL'),
    'S6-D4': ('TP9-TP7',  'L Inferior Temporal Gyrus (54%)', 'TEMPORAL'),
    'S7-D4': ('CP5-TP7',  'L Middle Temporal Gyrus (72%)',   'TEMPORAL'),
    'S7-D6': ('CP5-P5',   'L Middle Temporal Gyrus (52%)',   'TEMPORAL'),
    'S7-D7': ('T7-C5',    'L Middle Temporal Gyrus (55%)',   'TEMPORAL'),
    'S8-D4': ('T7-TP7',   'L Middle Temporal Gyrus (69%)',   'TEMPORAL'),
    'S8-D7': ('T7-C5',    'L Middle Temporal Gyrus (55%)',   'TEMPORAL'),
}

PALETTE = {
    'acelerado'    : 'darkorange',
    'nao_acelerado': 'steelblue'
}

print(f'Chromophore : {CHROMOPHORE}')
print(f'Channels    : {len(CHANNEL_MAP)} (Frontal: 7, Temporal: 10)')

## 2 — Load and prepare

In [ ]:
df_raw = pd.read_excel(PATH_IN)

# Filter chromophore
df = df_raw[df_raw['type'] == CHROMOPHORE].copy()

# Extract session number
def parse_session(exp):
    exp = str(exp).strip().lower()
    if 'calib' in exp: return 0
    m = re.search(r'(\d+)', exp)
    return int(m.group(1)) if m else -1

df['sessao_num'] = df['experiment'].apply(parse_session)

# Keep only mapped channels
df = df[df['channel'].isin(CHANNEL_MAP.keys())].copy()

# Add anatomical labels
df['name_1010']  = df['channel'].map(lambda c: CHANNEL_MAP[c][0])
df['region']     = df['channel'].map(lambda c: CHANNEL_MAP[c][1])
df['roi']        = df['channel'].map(lambda c: CHANNEL_MAP[c][2])

# Set reference categories
df['group'] = pd.Categorical(
    df['group'],
    categories=['nao_acelerado', 'acelerado'],
    ordered=False
)
df['subject'] = pd.Categorical(df['subject'])

print(f'Shape    : {df.shape}')
print(f'Subjects : {df["subject"].nunique()}  (expected 14)')
print(f'Sessions : {sorted(df["sessao_num"].unique())}')
print(f'Channels : {df["channel"].nunique()}  (expected {len(CHANNEL_MAP)})')
print(f'Missing  : {df["beta"].isna().sum()}')

## 3 — LMM per channel
`beta ~ group * sessao_num + (1|subject)` — Session 0 = calibration anchor

In [ ]:
FORMULA = 'beta ~ group * sessao_num'
results = []

for channel, (name_1010, region, roi) in CHANNEL_MAP.items():
    sub = df[df['channel'] == channel].copy()
    if sub['subject'].nunique() < 5:
        print(f'SKIP {channel} — insufficient subjects')
        continue
    try:
        lmm = smf.mixedlm(FORMULA, data=sub, groups=sub['subject'])
        res = lmm.fit(reml=False, method='bfgs')

        fe  = res.fe_params
        pv  = res.pvalues.loc[fe.index]
        ci  = res.conf_int().loc[fe.index]

        # Key interaction term
        inter_key = 'group[T.acelerado]:sessao_num'
        sess_key  = 'sessao_num'
        grp_key   = 'group[T.acelerado]'

        results.append({
            'channel'         : channel,
            'name_1010'       : name_1010,
            'region'          : region,
            'roi'             : roi,
            'chromophore'     : CHROMOPHORE,
            'converged'       : res.converged,
            # Group effect
            'coef_group'      : fe.get(grp_key, np.nan),
            'p_group'         : pv.get(grp_key, np.nan),
            # Session progression
            'coef_session'    : fe.get(sess_key, np.nan),
            'p_session'       : pv.get(sess_key, np.nan),
            # Key interaction
            'coef_interaction': fe.get(inter_key, np.nan),
            'p_interaction'   : pv.get(inter_key, np.nan),
            'ci_lower'        : ci.loc[inter_key, 0] if inter_key in ci.index else np.nan,
            'ci_upper'        : ci.loc[inter_key, 1] if inter_key in ci.index else np.nan,
        })
    except Exception as e:
        print(f'ERROR {channel}: {e}')

df_results = pd.DataFrame(results)
df_results['sig_interaction'] = df_results['p_interaction'] < 0.05
df_results['sig_session']     = df_results['p_session'] < 0.05

# Sort by ROI then p_interaction
df_results = df_results.sort_values(['roi','p_interaction']).reset_index(drop=True)

print(f'\nResults — {CHROMOPHORE}')
print('='*90)
print(df_results[['channel','region','roi',
                   'coef_interaction','p_interaction',
                   'ci_lower','ci_upper','sig_interaction']]
      .to_string(index=False))
print(f'\nSignificant interactions (p < 0.05): {df_results["sig_interaction"].sum()}')
print(df_results[df_results['sig_interaction']][['channel','region','roi','coef_interaction','p_interaction']]
      .to_string(index=False))

In [ ]:
from statsmodels.stats.multitest import multipletests

# FDR correction (Benjamini-Hochberg) on interaction p-values
pvals = df_results['p_interaction'].values
reject, pvals_fdr, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

df_results['q_interaction'] = pvals_fdr
df_results['sig_fdr']       = reject

print(f'FDR correction applied ({len(pvals)} tests)')
print(f'Significant after FDR (q < 0.05) : {reject.sum()}')
print(f'Significant uncorrected (p < 0.05): {df_results["sig_interaction"].sum()}')
print()
print('Results with FDR:')
print(df_results[['channel','region','roi',
                   'coef_interaction','p_interaction',
                   'q_interaction','sig_interaction','sig_fdr']]
      .to_string(index=False))

## 4 — Forest plot: interaction term by channel

In [ ]:
df_plot = df_results.copy()
# Label: region + channel
df_plot['label'] = df_plot['region'] + '\n(' + df_plot['channel'] + ' / ' + df_plot['name_1010'] + ')'

# Color by ROI and significance
def get_color(row):
    if row['sig_fdr']:
        return 'steelblue' if row['roi'] == 'FRONTAL' else 'darkorange'
    if row['sig_interaction']:  # significant uncorrected — hatched
        return 'lightblue' if row['roi'] == 'FRONTAL' else 'peachpuff'
    return 'lightgrey'

df_plot['color'] = df_plot.apply(get_color, axis=1)
df_plot = df_plot.iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, max(8, len(df_plot)*0.7)))

for i, row in df_plot.iterrows():
    c = row['color']
    ax.errorbar(
        y=i, x=row['coef_interaction'],
        xerr=[[row['coef_interaction'] - row['ci_lower']],
              [row['ci_upper'] - row['coef_interaction']]],
        fmt='o', color=c, capsize=4,
        markersize=8, markeredgecolor='black',
        ecolor=c, linewidth=1.5
    )
    p_str = f"p={row['p_interaction']:.3f}" if row['p_interaction'] >= 0.001 else 'p<0.001'
    ax.text(row['ci_upper'] + 0.005, i, p_str,
            va='center', fontsize=8,
            color='black' if row['sig_interaction'] else 'grey')

ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot['label'], fontsize=9)
ax.axvline(0, linestyle='--', color='grey', linewidth=1.2)
ax.set_title(
    f'Training-Induced Beta Change by Channel — {CHROMOPHORE}\n'
    f'(group × session interaction)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Coefficient β (Accelerated vs Non-Accelerated trajectory)', fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.5)

patches = [
    mpatches.Patch(color='steelblue',   label='Frontal — q < 0.05 (FDR)'),
    mpatches.Patch(color='darkorange',  label='Temporal — q < 0.05 (FDR)'),
    mpatches.Patch(color='lightblue',   label='Frontal — p < 0.05 only'),
    mpatches.Patch(color='peachpuff',   label='Temporal — p < 0.05 only'),
    mpatches.Patch(color='lightgrey',   label='n.s.'),
]
ax.legend(handles=patches, fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig(PATH_OUT_FIG, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {PATH_OUT_FIG}')
plt.show()

## 5 — Trajectory plot: significant channels only

In [ ]:
sig_channels = df_results[df_results['sig_fdr']]['channel'].tolist()

if not sig_channels:
    print('No significant channels to plot.')
else:
    n_sig = len(sig_channels)
    ncols = min(3, n_sig)
    nrows = int(np.ceil(n_sig / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5*ncols, 4*nrows),
                             squeeze=False)

    for idx, channel in enumerate(sig_channels):
        ax   = axes[idx // ncols][idx % ncols]
        info = CHANNEL_MAP[channel]
        sub  = df[df['channel'] == channel]

        traj = (
            sub.groupby(['sessao_num','group'], observed=True)['beta']
            .agg(['mean','sem'])
            .reset_index()
        )

        for grp, color, label in [
            ('acelerado',    'darkorange', 'Accelerated'),
            ('nao_acelerado','steelblue',  'Non-Accelerated')
        ]:
            s = traj[traj['group'] == grp].sort_values('sessao_num')
            ax.plot(s['sessao_num'], s['mean'],
                    marker='o', color=color, label=label, linewidth=2)
            ax.fill_between(
                s['sessao_num'],
                s['mean'] - s['sem'],
                s['mean'] + s['sem'],
                alpha=0.2, color=color
            )

        # Interaction p-value annotation
        row  = df_results[df_results['channel'] == channel].iloc[0]
        p_str = f"p={row['p_interaction']:.3f}" if row['p_interaction'] >= 0.001 else 'p<0.001'
        ax.annotate(f'Group×Session: {p_str}',
                    xy=(0.05, 0.95), xycoords='axes fraction',
                    fontsize=8, va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='white', edgecolor='grey', alpha=0.8))

        ax.axvline(0.5, linestyle=':', color='grey', linewidth=1, alpha=0.5)
        ax.axhline(0,   linestyle='--', color='grey', linewidth=0.8, alpha=0.4)
        ax.set_title(f'{info[1]}\n{channel} / {info[0]}',
                     fontsize=9, fontweight='bold')
        ax.set_xlabel('Session', fontsize=9)
        ax.set_ylabel(f'Beta ({CHROMOPHORE})', fontsize=9)
        ax.set_xticks(range(0, 10))
        ax.set_xticklabels(['C']+[str(i) for i in range(1,10)], fontsize=8)
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        if idx == 0:
            ax.legend(fontsize=8)

    # Hide unused axes
    for idx in range(n_sig, nrows*ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(
        f'Beta Trajectory — Significant Channels ({CHROMOPHORE})\n'
        f'Group × Session interaction p < 0.05',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(PATH_OUT_TRAJ, dpi=300, bbox_inches='tight')
    print(f'-> Figure saved: {PATH_OUT_TRAJ}')
    plt.show()

## 6 — Export

In [ ]:
with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    df_results.to_excel(writer, sheet_name='LMM_all_channels',  index=False)
    df_results[df_results['sig_interaction']].to_excel(
        writer, sheet_name='LMM_significant', index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print(f'   Chromophore: {CHROMOPHORE}')
print(f'   Channels tested: {len(df_results)}')
print(f'   Significant interactions: {df_results["sig_interaction"].sum()}')
print('   Sheets: LMM_all_channels | LMM_significant')